# 🎬 Piku vs Hyenas — 10-Second 3D Cartoon Generator (2 Clips)
### ⚡ Super Fast • Exactly 10 Seconds (5s + 5s) • With Hindi Voiceover!

**Runtime Check:** This notebook runs on **CPU or T4 GPU** (`Runtime -> Change runtime type`).

## 📦 Step 1: Install Dependencies & Download 2 Storyboard Scenes + Audio

In [ ]:
!pip install -q gradio_client ffmpeg-python
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Fresh clone of repository to fetch images & audio
!rm -rf videogen
!git clone https://github.com/yashkumarsahu789/videogen.git

import os
assets_dir = "videogen/video_output/piku_10s_slapstick/assets"
audio_file = "videogen/video_output/piku_10s_slapstick/audio/voiceover_10s_master.mp3"

print("✅ 2 Scene Storyboard Assets:", os.listdir(assets_dir))
print("✅ 10s Synchronized Voiceover:", os.path.exists(audio_file))

## 🚀 Step 2: Render Both Clips (5 Seconds Each via High-Speed Cloud Engine)

In [ ]:
import os, time, shutil
from pathlib import Path
from gradio_client import Client, handle_file

# Optional: If you have a free Hugging Face token, paste it here for unlimited quota (e.g. 'hf_...').
# If left empty, it runs using the free public cloud pool!
HF_TOKEN = None

print("🚀 Connecting to Lightricks LTX-Video Cloud Engine...")
client = Client("Lightricks/ltx-video-distilled", hf_token=HF_TOKEN)
print("✅ Connected to Cloud GPU!")

assets_path = Path("videogen/video_output/piku_10s_slapstick/assets")
output_dir = Path("output_clips")
output_dir.mkdir(parents=True, exist_ok=True)

clips = [
    ("clip1_piku_bowling_trap.jpg", "clip1.mp4", "Three panicked spotted hyenas sliding fast on giant green leaves screaming wide eyed, cheeky baby monkey laughing rolling spiky armadillo bowling ball downhill"),
    ("clip2_crocodile_crash_victory.jpg", "clip2.mp4", "Hyenas bounce into mossy log hanging upside-down dizzy with stars, baby monkey Piku peeks over rock with cheeky wink and double thumbs up")
]

clip_paths = []
for idx, (img_name, out_name, prompt) in enumerate(clips):
    img_file = assets_path / img_name
    out_file = output_dir / out_name
    print(f"\n🎬 Rendering Clip {idx+1}/2: {img_name}...")
    t0 = time.time()
    res = client.predict(
        prompt=prompt,
        negative_prompt="blurry, low quality, jittery, distorted",
        input_image_filepath=handle_file(str(img_file)),
        input_video_filepath=None,
        height_ui=512,
        width_ui=704,
        mode="image-to-video",
        duration_ui=5.0,
        ui_frames_to_use=9,
        seed_ui=42 + idx,
        randomize_seed=False,
        ui_guidance_scale=1,
        improve_texture_flag=True,
        api_name="/image_to_video"
    )
    shutil.copyfile(res[0]["video"], str(out_file))
    clip_paths.append(str(out_file))
    print(f"✓ Completed in {round(time.time() - t0, 1)}s -> {out_name}")

print("\n🎉 BOTH CLIPS RENDERED SUCCESSFULLY!")

## 🎶 Step 3: Concat & Add 10-Second Hindi Cartoon Voiceover

In [ ]:
assert len(clip_paths) == 2, f"Expected 2 clips, found {len(clip_paths)}"

with open("concat_list.txt", "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.abspath(cp)}'\n")

audio_path = "videogen/video_output/piku_10s_slapstick/audio/voiceover_10s_master.mp3"
final_output = "piku_10s_cartoon_master.mp4"

!ffmpeg -y -f concat -safe 0 -i concat_list.txt -i {audio_path} -c:v libx264 -c:a aac -shortest -pix_fmt yuv420p {final_output}
print(f"\n✅ Final 10-Second Movie with Hindi Voice Ready: {final_output}")

## 📥 Step 4: Preview & Download

In [ ]:
from IPython.display import HTML, FileLink
from base64 import b64encode

mp4_bytes = open(final_output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()

display(HTML(f'''
<div style="background:#1E293B; padding:20px; border-radius:16px; border:2px solid #10B981; max-width:760px; margin:auto; text-align:center;">
    <h3 style="color:#F59E0B; margin-top:0;">🐵 Piku vs Hyenas 10-Second Cartoon (With Voice)</h3>
    <video width=704 height=512 controls autoplay loop style="border-radius:12px; width:100%; border:2px solid #334155;">
        <source src="{data_url}" type="video/mp4">
    </video>
</div>
'''))

print("\n⬇️ Download Final MP4:")
display(FileLink(final_output))